# # Datalog

Demonstration of `IncrementalDatalog` on a small transitive-closure
program, and equivalence between batched and singleton-fed EDBs.

In [1]:
from pydbsp.algorithms.datalog import IncrementalDatalog, saturate
from pydbsp.algorithms import Variable
from pydbsp.history import History
from pydbsp.zset import ZSet

program = ZSet(
    {
        (
            ("T", (Variable("X"), Variable("Y"))),
            ("E", (Variable("X"), Variable("Y"))),
        ): 1,
        (
            ("T", (Variable("X"), Variable("Z"))),
            ("E", (Variable("X"), Variable("Y"))),
            ("T", (Variable("Y"), Variable("Z"))),
        ): 1,
    }
)

edges = [("E", (0, 1)), ("E", (1, 2)), ("E", (2, 3))]

## Batched evaluation

All facts pushed at `t=0`, a single `saturate` drives inner iteration.

In [2]:
circuit = IncrementalDatalog()
circuit.edb.push((0,), ZSet({e: 1 for e in edges}))
circuit.program.push((0,), program)
saturate(circuit)

history = History(circuit.observable)
history.try_step()
batched = history.at((0,))
print("batched T facts:", sorted(k[1] for k in batched.inner if k[0] == "T"))

batched T facts: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## Singleton-fed evaluation

Each fact pushed at its own outer tick; saturate per tick. Summed
outer-deltas must equal the batched result — that's the invariant
of the incremental feedback design.

In [3]:
circuit = IncrementalDatalog()
for t, fact in enumerate(edges):
    circuit.edb.push((t,), ZSet({fact: 1}))
    circuit.program.push((t,), program if t == 0 else ZSet({}))
    saturate(circuit, outer_tick=t)

history = History(circuit.observable)
cumulative: dict = {}
for t in range(len(edges)):
    history.try_step()
    for fact, weight in history.at((t,)).inner.items():
        cumulative[fact] = cumulative.get(fact, 0) + weight

print("singleton T facts:", sorted(k[1] for k in cumulative if k[0] == "T"))

singleton T facts: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
